# Reporte PDF para gerencia - Olist (versión final)
Ejecuta las celdas en orden (Shift+Enter).

## Paso 1: instalar librerías

In [ ]:
!pip install -q reportlab matplotlib pypdf

## Paso 2 (opcional): subir el PDF exportado de Power BI como anexo
Súbelo con el nombre `Olist_dashboard.pdf` (panel de archivos a la izquierda). Si no lo subes, el reporte se genera igual sin anexo.

## Paso 3: generar el PDF
Edita `AUTOR`, `TITULO` y las cifras si lo necesitas.

In [ ]:
# -*- coding: utf-8 -*-
"""
Reporte ejecutivo Olist (PDF) - funciona igual en Google Colab y en Spyder.
Requisitos:  pip install reportlab matplotlib pypdf
Salida:      Reporte_Olist_Gerencia.pdf (en la carpeta de trabajo)
"""
import os, datetime
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                TableStyle, Image, PageBreak, KeepTogether)

# ---------------------------------------------------------------- 1. CONFIGURACIÓN
TITULO   = "Análisis de Ventas, Logística y Clientes"
SUBTITULO = "Olist - E-commerce Brasil (sep. 2016 a ago. 2018)"
AUTOR    = "Hans Velásquez Leres"            # <- cambia por tu nombre
FECHA    = datetime.date.today().strftime("%d-%m-%Y")
SALIDA   = "Reporte_Olist_Gerencia.pdf"
ANEXO    = "Olist_dashboard.pdf"   # PDF exportado de Power BI (opcional; si no existe, se omite)
AZUL, ROJO, GRIS = "#1F5F7A", "#C0392B", "#95A5A6"

# ---------------------------------------------------------------- 2. DATOS (del dashboard Power BI)
KPIS = [("Ventas totales", "$13.221.498"), ("Pedidos", "96.478"),
        ("Ticket promedio", "$137,04"), ("Días de entrega (prom.)", "12,5"),
        ("Entregas tardías", "8,11 %"), ("Satisfacción (1-5)", "4,16"),
        ("Clientes únicos", "93.358"), ("Clientes que repiten", "3,0 %")]

ESTADOS = [("SP",5067633),("RJ",1759651),("MG",1552482),("RS",728897),("PR",666064),
           ("SC",507012),("BA",493584),("DF",296498),("GO",282837),("ES",268643)]
TRAMOS = [("8+ días antes",4.32,66129),("0-7 días antes",4.21,22034),("1-3 días tarde",3.77,2636),
          ("4-7 días tarde",2.32,1773),("8-14 días tarde",1.75,1748),("15+ días tarde",1.71,1504)]
TARDIAS = [("AL",23.93),("MA",19.67),("PI",15.97),("CE",15.32),("SE",15.22),
           ("BA",14.04),("RJ",13.47),("TO",12.77),("PA",12.37),("ES",12.23)]
RFM = [("Nuevos de alto valor",14483,15.5,3869193,29.3,267,92.2),
       ("En riesgo de alto valor",13784,14.8,3815846,28.9,277,395.1),
       ("Ocasionales",18070,19.4,2347553,17.8,130,220.5),
       ("Perdidos",22563,24.2,1258051,9.5,56,395.7),
       ("Nuevos",21657,23.2,1202446,9.1,56,90.1),
       ("Recurrentes",2801,3.0,728409,5.5,260,220.8)]
CATEG = [("health_beauty",1233132,8647,142.61,4.19),("watches_gifts",1166177,5495,212.23,4.07),
         ("bed_bath_table",1023435,9272,110.38,3.92),("sports_leisure",954853,7530,126.81,4.17),
         ("computers_accessories",888725,6530,136.10,3.99),("furniture_decor",711928,6307,112.88,3.95),
         ("housewares",615629,5743,107.20,4.11),("cool_stuff",610204,3559,171.45,4.19),
         ("auto",578967,3810,151.96,4.12),("toys",471286,3804,123.89,4.21)]

MESES = [("2017-01",111798),("2017-02",234223),("2017-03",359199),("2017-04",340670),("2017-05",489338),
         ("2017-06",421923),("2017-07",481605),("2017-08",554700),("2017-09",607400),("2017-10",648248),
         ("2017-11",987765),("2017-12",726033),("2018-01",924645),("2018-02",826437),("2018-03",953356),
         ("2018-04",973534),("2018-05",977545),("2018-06",856078),("2018-07",867953),("2018-08",838577)]
# Traducción de categorías (el dataset original las trae en inglés)
TRAD = {"health_beauty":"Salud y belleza","watches_gifts":"Relojes y regalos","bed_bath_table":"Cama, baño y mesa",
        "sports_leisure":"Deportes y ocio","computers_accessories":"Accesorios de computación",
        "furniture_decor":"Muebles y decoración","housewares":"Artículos del hogar","cool_stuff":"Artículos novedosos",
        "auto":"Automotriz","toys":"Juguetes","office_furniture":"Muebles de oficina"}
CATEG = [(TRAD.get(c[0], c[0]),) + c[1:] for c in CATEG]
PEDIDOS_TARDIOS = 7826     # 8,11 % de 96.478 pedidos entregados

def miles(n): return f"{n:,.0f}".replace(",", ".")

# ---------------------------------------------------------------- 3. GRÁFICOS (matplotlib -> PNG)
os.makedirs("graficos", exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def barh(nombre, etiquetas, valores, color, titulo, fmt, resaltar=None):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    cols = [ROJO if (resaltar and e in resaltar) else color for e in etiquetas]
    ax.barh(etiquetas[::-1], valores[::-1], color=cols[::-1])
    for i, v in enumerate(valores[::-1]):
        ax.text(v, i, " " + fmt(v), va="center", fontsize=8)
    ax.set_title(titulo, loc="left", fontweight="bold")
    ax.xaxis.set_visible(False); ax.spines["bottom"].set_visible(False)
    fig.tight_layout(); fig.savefig(f"graficos/{nombre}.png", dpi=200); plt.close(fig)

barh("estados", [e for e,_ in ESTADOS], [v for _,v in ESTADOS], AZUL,
     "Top 10 estados por ventas ($)", lambda v: "$"+miles(v))
barh("tardias", [e for e,_ in TARDIAS], [v for _,v in TARDIAS], ROJO,
     "% de entregas tardías: 10 estados con mayor tasa", lambda v: f"{v:.1f} %".replace(".",","))
barh("rfm", [r[0] for r in RFM], [r[4] for r in RFM], AZUL,
     "Peso en ventas por segmento de clientes (%)", lambda v: f"{v:.1f} %".replace(".",","),
     resaltar=["En riesgo de alto valor"])
barh("categorias", [c[0] for c in CATEG], [c[1] for c in CATEG], AZUL,
     "Top 10 categorías por ventas ($)", lambda v: "$"+miles(v))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot([m[0] for m in MESES], [m[1]/1000 for m in MESES], color=AZUL, marker="o", ms=3)
ax.set_xticks(range(0, len(MESES), 2)); ax.set_xticklabels([m[0] for i, m in enumerate(MESES) if i % 2 == 0], rotation=45, fontsize=7)
ax.set_ylabel("Miles de $"); ax.set_title("Ventas mensuales (miles de $)", loc="left", fontweight="bold")
ax.annotate("Nov-2017: $987.765", xy=(10, 987.765), xytext=(2, 900), fontsize=8, arrowprops=dict(arrowstyle="->", color=GRIS))
fig.tight_layout(); fig.savefig("graficos/mensual.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
et = [t[0] for t in TRAMOS]; val = [t[1] for t in TRAMOS]
ax.bar(et, val, color=[AZUL,AZUL,AZUL,ROJO,ROJO,ROJO])
for i, v in enumerate(val): ax.text(i, v+0.05, f"{v:.2f}".replace(".",","), ha="center", fontsize=8)
ax.set_title("Satisfacción promedio según días de retraso", loc="left", fontweight="bold")
ax.set_ylim(0, 5); plt.xticks(rotation=20)
fig.tight_layout(); fig.savefig("graficos/satisfaccion.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar([t[0] for t in TRAMOS], [t[2] for t in TRAMOS], color=[AZUL,AZUL,AZUL,ROJO,ROJO,ROJO])
for i, t_ in enumerate(TRAMOS): ax.text(i, t_[2], miles(t_[2]), ha="center", va="bottom", fontsize=8)
ax.set_title("Pedidos por tramo de entrega", loc="left", fontweight="bold")
ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); plt.xticks(rotation=20)
fig.tight_layout(); fig.savefig("graficos/tramos.png", dpi=200); plt.close(fig)

# ---------------------------------------------------------------- 4. ESTILOS
ss = getSampleStyleSheet()
H1  = ParagraphStyle("H1", parent=ss["Heading1"], textColor=colors.HexColor(AZUL), fontSize=16, spaceAfter=6)
H2  = ParagraphStyle("H2", parent=ss["Heading2"], textColor=colors.HexColor(AZUL), fontSize=12, spaceBefore=8, spaceAfter=4)
TXT = ParagraphStyle("TXT", parent=ss["BodyText"], fontSize=10, leading=14, alignment=TA_JUSTIFY)
BUL = ParagraphStyle("BUL", parent=TXT, leftIndent=12, bulletIndent=2, spaceAfter=2)
NOTA= ParagraphStyle("NOTA", parent=TXT, fontSize=8, textColor=colors.HexColor("#555555"))
def P(t, s=TXT): return Paragraph(t, s)
def B(t): return Paragraph(t, BUL, bulletText="•")
def img(n, w=15.5): 
    return Image(f"graficos/{n}.png", width=w*cm, height=w*cm*3.2/6.5)

def tabla(datos, anchos, cabecera_color=AZUL):
    t = Table(datos, colWidths=anchos, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,0),colors.HexColor(cabecera_color)),
        ("TEXTCOLOR",(0,0),(-1,0),colors.white), ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),
        ("FONTSIZE",(0,0),(-1,-1),8.5), ("ALIGN",(1,0),(-1,-1),"RIGHT"),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.white, colors.HexColor("#F2F6F8")]),
        ("GRID",(0,0),(-1,-1),0.25,colors.HexColor("#CCCCCC")),
        ("TOPPADDING",(0,0),(-1,-1),3), ("BOTTOMPADDING",(0,0),(-1,-1),3)]))
    return t

# ---------------------------------------------------------------- 5. PORTADA, PIE Y CONTENIDO
def pie(canvas, doc):
    canvas.saveState(); canvas.setFont("Helvetica", 8); canvas.setFillColor(colors.grey)
    canvas.drawString(2*cm, 1.2*cm, f"{TITULO} | {FECHA}")
    canvas.drawRightString(A4[0]-2*cm, 1.2*cm, f"Página {doc.page}")
    canvas.restoreState()

def portada(canvas, doc):
    w, h = A4
    canvas.saveState()
    canvas.setFillColor(colors.HexColor(AZUL)); canvas.rect(0, h*0.52, w, h*0.48, fill=1, stroke=0)
    canvas.setFillColor(colors.white); canvas.setFont("Helvetica-Bold", 28)
    canvas.drawString(2*cm, h*0.80, "Reporte Ejecutivo")
    canvas.setFont("Helvetica-Bold", 20); canvas.drawString(2*cm, h*0.74, TITULO)
    canvas.setFont("Helvetica", 13);      canvas.drawString(2*cm, h*0.69, SUBTITULO)
    canvas.setFillColor(colors.HexColor("#333333")); canvas.setFont("Helvetica", 12)
    canvas.drawString(2*cm, h*0.44, f"Fecha: {FECHA}")
    canvas.drawString(2*cm, h*0.41, f"Elaborado por: {AUTOR}")
    canvas.drawString(2*cm, h*0.38, "Dirigido a: Gerencia")
    canvas.restoreState()

story = [Spacer(1, 1), PageBreak()]

# --- Introducción
story += [P("1. Introducción", H1),
 P("Este informe resume los resultados del dashboard de Power BI construido sobre el conjunto de datos público "
   "de Olist (marketplace brasileño), con <b>96.478 pedidos entregados</b> entre septiembre de 2016 y agosto de 2018. "
   "El objetivo es entregar a la gerencia una visión breve de cuatro ejes: <b>ventas, productos, logística y clientes</b>, "
   "para apoyar la toma de decisiones."),
 P("Metodología: los datos se limpiaron en Power Query y Python, se consolidaron en MySQL y se modelaron en Power BI "
   "(medidas DAX por pedido). Los indicadores consideran solo pedidos entregados. La satisfacción es el promedio de la "
   "nota de reseña (1 a 5) por pedido."),
 Spacer(1, 6)]

# --- KPIs
story.append(P("2. Hallazgos destacados (KPIs)", H1))
celdas = [[Paragraph(f"<font size=8 color='#555555'>{k}</font><br/><font size=15 color='{ROJO if 'tard' in k else AZUL}'><b>{v}</b></font>", TXT)
           for k, v in KPIS[i:i+4]] for i in (0, 4)]
tk = Table(celdas, colWidths=[4.25*cm]*4, rowHeights=[1.7*cm]*2)
tk.setStyle(TableStyle([("BOX",(0,0),(-1,-1),0.5,colors.HexColor("#CCCCCC")),
                        ("INNERGRID",(0,0),(-1,-1),0.5,colors.HexColor("#CCCCCC")),
                        ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F2F6F8")),
                        ("VALIGN",(0,0),(-1,-1),"MIDDLE")]))
story += [tk, Spacer(1, 8)]

story += [P("2.1 Ventas", H2),
 B("Las ventas se concentran en el sureste: <b>SP 38 %</b> del total; SP, RJ y MG suman el <b>63 %</b>."),
 B("Las ventas mensuales crecieron durante 2017 y se estabilizaron en torno a $0,83-0,98 millones desde 2018."),
 B("El máximo mensual fue noviembre de 2017 (<b>$987.765</b>); coincide con Black Friday, hipótesis que no se verificó con los datos."),
 img("mensual", 11.5)]

story += [img("estados", 11), Spacer(1,4), P("2.2 Productos", H2),
 B("Las 3 primeras categorías (salud y belleza, relojes y regalos, cama/baño/mesa) suman el <b>26 %</b> de las ventas; las 10 primeras, el <b>62 %</b>."),
 B("Cama/baño/mesa lidera en pedidos (9.272) con un ticket bajo ($110); relojes y regalos y muebles de oficina tienen los tickets más altos."),
 B("Las notas más bajas están en muebles de oficina (3,51), cama/baño/mesa (3,92) y muebles y decoración (3,95)."),
 img("categorias", 11.5)]
t = [["Categoría","Ventas ($)","Pedidos","Ticket ($)","Satisf."]] + \
    [[c[0], miles(c[1]), miles(c[2]), f"{c[3]:.2f}".replace(".",","), f"{c[4]:.2f}".replace(".",",")] for c in CATEG]
story += [tabla(t, [5*cm,3.2*cm,2.6*cm,2.6*cm,2.2*cm]), Spacer(1,6),
          P("Nota: la satisfacción por categoría es el promedio por ítem vendido (4,08 global por ítem vs. 4,16 por pedido).", NOTA),
          PageBreak()]

story += [P("2.3 Logística y satisfacción", H2),
 B("El tiempo medio de entrega es de <b>12,5 días</b> (mediana 10) y el <b>8,11 %</b> de los pedidos llegó tarde."),
 B("La satisfacción cae de <b>4,2</b> (a tiempo) a <b>2,3</b> con 4 a 7 días de retraso y a <b>1,7</b> con más de una semana. "
   "Llegar antes casi no la mejora (4,32 vs 4,21)."),
 B("Los estados del nordeste (AL, MA, PI, CE, SE, BA) presentan las mayores tasas de entregas tardías; AL llega al 23,9 %."),
 B("En términos absolutos, hay unos <b>7.800 pedidos</b> entregados con retraso: es donde se concentra la insatisfacción."),
 img("satisfaccion", 10), img("tramos", 10), img("tardias", 11),
 P("Estados con al menos 100 pedidos. Base: 95.824 pedidos con reseña y fecha de entrega.", NOTA), PageBreak()]

story += [P("2.4 Clientes (segmentación RFM)", H2),
 B("Los dos segmentos de alto valor son el <b>30 %</b> de los clientes y generan el <b>58 %</b> de las ventas."),
 B("<b>13.784 clientes «En riesgo de alto valor»</b> llevan unos 395 días sin comprar. Históricamente compraron <b>$3,8 millones</b> (29 % de las ventas), por lo que son la primera oportunidad de reactivación; el monto es lo ya comprado, no lo que se recuperaría."),
 B("Solo el <b>3 %</b> de los clientes compró más de una vez; el gasto promedio por cliente es $142 (distinto del ticket por pedido, $137,04)."),
 img("rfm", 12)]
t = [["Segmento","Clientes","% Cli.","Ventas ($)","% Vtas","Gasto prom.","Recencia"]] + \
    [[r[0], miles(r[1]), f"{r[2]:.1f} %".replace(".",","), miles(r[3]), f"{r[4]:.1f} %".replace(".",","),
      "$"+str(r[5]), f"{r[6]:.1f}".replace(".",",")] for r in RFM]
story += [tabla(t, [4.2*cm,2*cm,1.7*cm,2.6*cm,1.7*cm,2.2*cm,2*cm]), Spacer(1,6),
          P("«En riesgo» y «Perdidos» dependen de que los datos terminan en agosto de 2018.", NOTA), PageBreak()]

# --- Conclusiones
story += [P("3. Conclusiones y recomendaciones", H1),
 P("<b>Conclusiones</b>"),
 B("El negocio depende fuertemente del sureste (SP, RJ y MG) y de pocas categorías."),
 B("La puntualidad es un factor fuertemente asociado a la satisfacción: las entregas tardías (8 %, unos 7.800 pedidos) concentran las peores notas, mientras que llegar antes casi no mejora la nota."),
 B("La recompra es muy baja (3 %), por lo que la base de clientes es en su mayoría de compra única."),
 Spacer(1,4), P("<b>Recomendaciones a evaluar</b>"),
 B("Atacar las entregas tardías en los estados con mayor tasa (AL, MA, PI, CE, SE, BA) revisando transportistas y plazos prometidos."),
 B("Campaña de reactivación dirigida a los 13.784 clientes «En riesgo de alto valor»."),
 B("Revisar calidad y logística en categorías con notas bajas (muebles de oficina, cama/baño/mesa, muebles y decoración)."),
 B("Diseñar acciones de recompra para convertir clientes nuevos en recurrentes."),
 Spacer(1,8),
 P("<b>Alcance y limitaciones</b>"),
 P("Este informe se elaboró con los datos disponibles a la fecha de emisión (sep. 2016 a ago. 2018). "
   "Si se eliminan o se incorporan nuevos datos, los indicadores, gráficos y conclusiones del informe estarán sujetos a cambios "
   "y deberán actualizarse. Las asociaciones descritas (por ejemplo, retraso y satisfacción) no implican causalidad."),
 Spacer(1,6),
 P("Fuente: Olist Brazilian E-Commerce Public Dataset (Kaggle). Elaboración propia.", NOTA)]

# ---------------------------------------------------------------- 6. GENERAR PDF
doc = SimpleDocTemplate(SALIDA, pagesize=A4, leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=2*cm, bottomMargin=2*cm, title=TITULO, author=AUTOR)
doc.build(story, onFirstPage=portada, onLaterPages=pie)

# Anexo opcional: páginas del dashboard exportadas desde Power BI
if os.path.exists(ANEXO):
    from pypdf import PdfWriter
    w = PdfWriter()
    w.append(SALIDA); w.append(ANEXO)
    w.write(SALIDA); w.close()
    print("Anexo agregado:", ANEXO)
print("PDF generado:", os.path.abspath(SALIDA))


## Paso 4: descargar

In [ ]:
from google.colab import files
files.download('Reporte_Olist_Gerencia.pdf')